In [0]:
import requests
import json
import boto3

In [0]:
def pn_api_extraction(api_url,api_param,headers,offset,limit):

    all_barca_players = []    
    
    while True:
        api_param['limit'] = limit
        api_param['offset'] = offset

        res = requests.get(api_url, params=api_param, headers=headers)
        data = res.json()
       
        if top_lvl_filter is not None:
            data = data.get(top_lvl_filter, [])

        # Stop when no more data
        if not data and isinstance(data, dict):
            break
        if len(data) == 0 and isinstance(data, list):
            break
        for p in data:
            all_barca_players.append(p)

        offset += limit 
    json_data = json.dumps(all_barca_players, indent=4, ensure_ascii=False)
    
    return json_data

def api_extraction(api_url,api_param):

    all_barca_players = []

    response = requests.get(api_url, params=api_param)
    json_data = response.json()

    if top_lvl_filter is not None:
        json_data = json_data.get(top_lvl_filter, [])
    
    json_data = json.dumps(json_data, indent=4, ensure_ascii=False)
    
    return json_data

In [0]:
def upload_s3_vol(json_data,bucket_name,s3_key,volume_path):
    s3_error_message = {}
    dbk_error_message = {}

    try:
        # Create S3 client
        s3 = boto3.client(
            "s3",
            aws_access_key_id=access_key,
            aws_secret_access_key=secret_key,
            region_name="us-east-1"
        )

        # Upload to S3
        s3.put_object(
            Bucket=bucket_name,
            Key=s3_key,
            Body=json_data,
            ContentType="application/json"
        )
        print(f"File uploaded to S3 the path:{s3_key}")
    except Exception as e:
        s3_error_message[file_name] = f"Error uploading file to S3: {str(e)}"
       
    try:
        response = requests.put(
            f"{databricks_host}/api/2.0/fs/files{volume_path}",
            headers={
                "Authorization": f"Bearer {databricks_token}",
                "Content-Type": "application/octet-stream"
            },
            data=json_data.encode("utf-8")
        )

        print(f"File uploaded to volume {volume_path}")
    except Exception as e:
        dbk_error_message[file_name] = f"Error uploading file to volume: {str(e)}"

In [0]:
def raw_file_to_table(data,file_name):
    
    selected_data = []
    col_len = len(list(data[0].keys()))
    ref_col = list(data[0].keys())

    for row in data:
        record = {}
        columns = list(row.keys())
        if(len(columns) > col_len):
            ref_col = columns

        for col in columns:
            value = row.get(col)

            if isinstance(value, (dict, list)):
                value = json.dumps(value, ensure_ascii=False)

            record[col] = value

        selected_data.append(record)

    df_file = spark.createDataFrame(selected_data).select(ref_col)
    
    (
        df_file.write.mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"workspace.bk_raw.{file_name}")
    )